# 06 Explain and answer

**Owner:** Person C. This notebook is also where the three of you meet again at 2:20.

**Inputs:** `outputs/models/selected_model.joblib`, `data/processed/model_matrix.parquet`, `splits/split.npz`, `outputs/answers.md`.

**Outputs:** `outputs/tables/gain_importance.csv`, `outputs/tables/permutation_importance.csv`, `outputs/tables/shap_values.csv`, `outputs/figures/importance_top15.png`.

**Stop rule:** if no model was saved, skip SHAP and write the answers from the profiles, the match rate, and the two-way table. Do not overwrite `outputs/answers.md` once someone has started filling it in. At 2:10, an unstarted SHAP run is dropped.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.paths import (
    RAW,
    PROCESSED,
    OUTPUTS,
    TABLES,
    ANALYSIS_PATH,
    FEATURES_PATH,
    MODEL_MATRIX_PATH,
    ROLES_PATH,
    CONTRACT_PATH,
    GATE_PATH,
    FEATURE_LOG_PATH,
    ALIAS_PATH,
    ANSWERS_PATH,
    COMPARISON_PATH,
    BUNDLE_PATH,
    ensure_dirs,
)

ensure_dirs()
print("root", ROOT)


In [ ]:
import numpy as np

from src import io as sio
from src.explain import write_explanations
from src.models import load_bundle
from src.split import load_split, split_exists

print("Answers file:", ANSWERS_PATH)
print("Fill it at 2:20. This notebook does not overwrite it.")
if not BUNDLE_PATH.exists() or not split_exists():
    print("STOP: no saved model or no frozen split. SHAP is skipped.")
elif not MODEL_MATRIX_PATH.exists() and not MODEL_MATRIX_PATH.with_suffix(".csv").exists():
    print("STOP: model matrix is missing.")
else:
    bundle = load_bundle(BUNDLE_PATH)
    frame = sio.read_frame(MODEL_MATRIX_PATH).reset_index(drop=True)
    split = load_split()
    val = frame.iloc[split["val"]]
    mask = val[bundle["target"]].notna().to_numpy()
    transformed = np.asarray(bundle["preprocessor"].transform(val), dtype=float)[mask]
    labels = val[bundle["target"]].to_numpy()[mask]
    if len(transformed) == 0:
        print("STOP: validation has no labelled rows.")
    else:
        paths = write_explanations(
            bundle["model"],
            transformed,
            labels,
            bundle["feature_names"],
            bundle["task"],
        )
        for name, path in paths.items():
            print(name, path)
